# Favorita daily sales forecasting

Start with the prediction task, input keys and temporal split. This notebook documents the selected seven-day-token, three-year-history Transformer. Recorded test WAPE is 49.9963%.

In [ ]:
from pathlib import Path
import json, sys, subprocess
ROOT = Path.cwd()
if ROOT.name == "favorita":
    ROOT = ROOT.parent
assert (ROOT / "favorita" / "train.py").exists()
sys.path.insert(0, str(ROOT / "favorita"))

## Prepare the data

Obtain the competition train.csv using the README link. Install CUDA-enabled PyTorch and favorita/requirements.txt. The preparation step aggregates store, product and date, preserving chronological order.

In [ ]:
TRAIN_CSV = ROOT / "data/favorita/train.csv"
PANEL = ROOT / "work/favorita-panel.parquet"
# Run once after downloading:
# subprocess.run([sys.executable, str(ROOT / "favorita/prepare.py"),
#                 "--train", str(TRAIN_CSV)], cwd=ROOT, check=True)

## Inspect input schema

Negative aggregate sales are clipped to zero during loading. Missing rows after series launch become zero. Earlier dates are masked.

In [ ]:
import pyarrow.parquet as pq
if PANEL.exists():
    panel = pq.ParquetFile(PANEL)
    print(panel.schema_arrow)
    print("Rows:", panel.metadata.num_rows)
    display(next(panel.iter_batches(batch_size=5)).to_pandas())
else:
    print("Prepare the panel to inspect local data.")

## Define the experiment

Fit targets end June 20, 2017. Validation is June 21–July 18. Test is July 19–August 15. Each forecast uses actual sales through the previous day, including previous actual holdout observations. This is rolling one-day-ahead evaluation. Fit target eligibility begins at max(28, first observation + 1).

In [ ]:
import pandas as pd
display(pd.DataFrame([["Fit", "2013-01-01", "2017-06-20"],
                      ["Validation", "2017-06-21", "2017-07-18"],
                      ["Test", "2017-07-19", "2017-08-15"]],
                     columns=["Split", "Panel/window start", "End"]))

## Construct the model

157 tokens represent 1,099 historical days. Each seven-day token preserves its daily values through concatenation and projection. Width 128, 12 layers, 8 heads and global historical attention.

In [ ]:
import torch
from retail_patch import PatchTransformer
torch.manual_seed(42)
model = PatchTransformer(7,174685,True,7,sales_history=True,patch_count=157,layers=12)
parameters = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {parameters:,}")
assert parameters == 7240641

## Run full training

All parameters train for two epochs. AdamW learning rate 0.001, batch 1,024, BF16 autocast, L1 loss. Validation WAPE selects weights. Full training can take multiple days. Set the flag deliberately.

In [ ]:
RUN_FULL_TRAINING = False
if RUN_FULL_TRAINING:
    subprocess.run([sys.executable, str(ROOT / "favorita/train.py"),
                    "favorita", "long"], cwd=ROOT, check=True)

## Inspect recorded results

WAPE sums absolute errors over all scored item-store-day rows and divides by total actual sales. These metrics and plots come from the completed run. Running these cells reads the bundled artifacts.

In [ ]:
metrics = json.loads((ROOT / "favorita/metrics.json").read_text())
print("Validation WAPE", format(metrics["validation_wape"], ".4%"))
print("Test WAPE", format(metrics["test_wape"], ".4%"))
display(pd.DataFrame([metrics["surge"],metrics["drop"]],index=["Upward","Downward"]))

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "favorita/forecast.png")))

## Interpretation

The recorded score is the best completed full-panel Favorita run checked for publication. The notebook does not execute full training by default. Independent full retraining is needed to establish reproduction on another GPU/software stack.